# Linear and non-linear regression with hyperparameter/feature selection tuning optimization (using Optuna) and K-fold cross-validation

Refer to https://github.com/SigmanGroup/BO_modeling for original script

## 1. Imports

In [20]:
# stdlib
import re
from pathlib import Path
from tqdm import tqdm
import datetime
run_date = datetime.datetime.now().strftime("%b%d")

# Data wrangling
import numpy as np
import pandas as pd

# ML models and metrics
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, BayesianRidge)
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from utils_github import train_outer_fold, safe_rmse

# Visualization
import matplotlib.pyplot as plt
import shap
import matplotlib as mpl
import seaborn as sns
plt.rcParams["font.size"] = 8
import textwrap
from scipy.stats import linregress
from ast import For
import itertools
from matplotlib.colors import LinearSegmentedColormap

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 2. General Parameters
Modifications required by user to specify details for your dataset.
This cell specifies path for data input, column names, results directory, number of inner and outer splits, number of model features, etc.

In [ ]:
# Input spreadsheet, sheet name, target column, ID columns for your dataset, & drop unneeded column names 
excel_file = Path("./InputData/Data.xlsx")
sheet_name = "Sheet1"
target_column = "ln(r)"
id_columns = ["Experiment"] 
drop_columns = ["M1", "M2", "r value"]

# Define name and location of output files
results_dir = Path(f'./results/{run_date}')
results_dir.mkdir(exist_ok=True)
output_prefix = "CyclicOlefinProject"

# Define number of outer splits, times repeated in the workflow, inner splits, and TFE B.O. trials
# Larger number of splits/trials dramatically increases compute time
outer_n_splits = 4
outer_n_repeats = 3
inner_cv_splits = 4
inner_cv_repeats = 3
n_trials = 20

# Minimum and maximum number of model features & feature correlation threshold
min_features = 4 
max_features_cap = 10 
corr_threshold = 0.75

# Define overfitting penalty in the objective function, higher value will penalize overfitting more
# If (Train - Val) = overfit_penalty > 0, good
# If (Train - Val) = overfit_penalty ≤ 0, replace it with 0
# Final score = Val - (overfit_penalty_weight * (Train - Val))
overfit_penalty_weight = 1

# Keep consistent random state for reproducibility
random_state=42

# Define which model types to be used in the BO campaign 
# bo_model_names = ["MLR", "Ridge", "Lasso", "BRR", "KRR", "SVR", "KNN", "RF", "XGB"]
bo_model_names = ["KRR"]

## 3. Load And Clean The Data
Do not modify this cell. The matrix `X` contains the features, and `y` contains the continuous target variable.

In [ ]:
# Read in the data and drop columns we don't want and NaN columns
df = pd.read_excel(excel_file,sheet_name=sheet_name)
df = df.drop(columns=drop_columns, errors="ignore")
df = df.drop(columns=[col for col in id_columns if col in df.columns], errors="ignore")
df = df.dropna(subset=[target_column])

# Get X and y
X = df.drop(columns=[target_column])
y = df[target_column]
X = X.apply(pd.to_numeric,errors="coerce")

# Get the valid rows with no missing features at all
valid_rows = X.notna().all(axis=1)

# Keep only the clean data
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

# Shows the adjusted number of rows and columns in your dataframe
print(f"Loaded dataset: {X.shape[0]} rows, {X.shape[1]} features.")

## 4. Correlation matrix
No modifications needed here, caution when using this if you have a large number of features



In [ ]:
# Creates a plot of this size
plt.figure(figsize=(15, 15))

# Actually creates the heatmap matrix
sns.heatmap(X.corr(),cmap="Spectral",linewidths=0.5,annot=True)

# Adjusts spacing so we can see everything
plt.tight_layout()

# Saves figure in results directory
plt.savefig(results_dir / f"correlation_heatmap_{output_prefix}.svg")

# Closes the plot from memory to free up resources
plt.close()

## 5. Outer Cross Validation
No modifications needed here, this cell sets up the proper outer CV to run



In [ ]:
# Creates CV splitter
outer_cv = RepeatedKFold(n_splits=outer_n_splits,n_repeats=outer_n_repeats,random_state=42)

# Creates a label string
outer_name = f"{outer_n_splits}x{outer_n_repeats}RepeatedKFold"

## 6. Nested Training and Validation
This cell should not be modified. For each model and each outer fold:
1. Define an outer train/test split (outer CV fold)
2. Run Optuna TPE (B.O.) on inner CV where each trial tunes model hyperparameters and feature selection method/parameters
3. Optionally penalize overfitting
4. Select best Optuna configuration and retrain final pipeline on full outer training set
5. Evaluate performance on outer test fold
6. Store metrics, predictions, parameters, and selected features

In [ ]:
# Store all ensemble models
class EnsembleModel:
    def __init__(self, models):
        self.models = models

    #Make predictions by averaging predictions from all fold models (ensemble)
    def predict(self, X_input):
        preds = []
        for model in self.models:
            preds.append(model.predict(X_input))
        return np.column_stack(preds).mean(axis=1)

# Storage containers for results across ALL model types and folds
summary_results = [] # One final score per model after all folds are completed
all_predictions = [] # Every test prediction across all folds and models
all_params = [] # Hyperparameters and performance metrics for each fold and model
all_selected_features = [] # Features chosen per fold and model

# Dictionaries for the final best models and features
models_dict = {} 
features_dict_final = {}

# Looping over all model types 
for model_name in bo_model_names:
    print(f"\nStarting {model_name}")
    y_true_all = []
    y_pred_all = []
    model_folds = []
    fold_params = []

    # Outer cross-validation loop
    # for fold, (train_idx, test_idx) in enumerate(tqdm(outer_cv.split(X), total=outer_n_splits*outer_n_repeats, desc=f"Outer CV Fold:"), start=1):
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X),start=1):
        print(f"\tStarting Fold {fold}")
        
        # Train/test splits for outer fold
        X_train_outer = X.iloc[train_idx]
        X_test_outer = X.iloc[test_idx]
        y_train_outer = y.iloc[train_idx]
        y_test_outer = y.iloc[test_idx]

        # Train model on outer fold 
        result = train_outer_fold(model_name=model_name,
            X_train_outer=X_train_outer,
            y_train_outer=y_train_outer,
            X_test_outer=X_test_outer,
            y_test_outer=y_test_outer,
            inner_cv_splits=inner_cv_splits,
            inner_cv_repeats=inner_cv_repeats,
            max_features_cap=max_features_cap,
            corr_threshold=corr_threshold,
            n_trials=n_trials,
            overfit_penalty_weight=overfit_penalty_weight,
            random_state=random_state)

        # fullracting outputs
        pred = result["pred_test"]
        best_features = result["selected_features"]
        best_selector = result["feature_selection_method"]
        metrics = result["metrics"]
        params = result["best_params"]

        # Store hyperparameters and metrics for this fold
        fold_params.append({"params": params, "r2": metrics["r2_test"]})

        # Store predictions 
        for i, (true_value, pred_value) in enumerate(zip(y_test_outer.values, pred)):
            all_predictions.append({"Model": model_name, "Fold": fold, "Sample_ID": int(test_idx[i]), "True": true_value, "Predicted": pred_value, "Residual": true_value - pred_value})
            y_true_all.append(true_value)
            y_pred_all.append(pred_value)

        # Store all hyperparameters and metrics 
        all_params.append({"Model": model_name, "Fold": fold, **metrics, "Params": str(params), "Feature_Selection_Method": best_selector, "Features": ", ".join(best_features)})

        # Store selected features for this fold
        for feat in best_features: all_selected_features.append({"Model": model_name, "Fold": fold, "Feature_Selection_Method": best_selector, "Feature": feat})

        # Store the trained model pipeline, metrics, and features for this fold    
        model_folds.append({"Model": model_name, "Fold": fold, "Pipeline": result["model"], "R2_test": metrics["r2_test"], "Features": best_features})

    # CV summary statistics for this model type
    fold_r2s = [f["R2_test"] for f in model_folds]
    mean_r2 = np.mean(fold_r2s)
    std_r2 = np.std(fold_r2s)

    # Create ensemble model list
    fold_models = [f["Pipeline"] for f in model_folds]
    features_per_fold = [f["Features"] for f in model_folds]

    all_fold_models = [] # Stores the trained pipeline, features, and hyperparameters for each fold and model

    # Model evaluation using all outer fold predictions
    summary_results.append({
        "Model": model_name,
        "Mean_R2_CV": mean_r2,
        "Std_R2_CV": std_r2,
        "R2_all_predictions": r2_score(y_true_all, y_pred_all),
        "RMSE_all_predictions": safe_rmse(y_true_all, y_pred_all),
        "MAE_all_predictions": mean_absolute_error(y_true_all, y_pred_all)})

    # Store ensemble model (fixes lambda closure bug and dict structure inconsistency)
    models_dict[model_name] = EnsembleModel(fold_models)

    # Store primary feature set for reference (from first fold)
    # Note: Each fold may have different features; the EnsembleModel handles this internally
    from collections import Counter
    all_feats = [feat for fold in model_folds for feat in fold["Features"]]
    features_dict_final[model_name] = [f for f, _ in Counter(all_feats).most_common(max_features_cap)]

## 7. Combining All Results
This cell should not be modified. 
This section aggregates results across all models and outer cross validation folds to evalute predictive performance and model behavior
The main metrics are:
`R²`: proportion of variance explained
`RMSE`: root mean squared error (in the target unit)
`MAE`: mean absolute error (more robust to large errors)

In [ ]:
# Summary table from above results
df_summary_nested = pd.DataFrame(summary_results).sort_values(by="R2_all_predictions",ascending=False)

# Raw predictions table
df_predictions_nested = pd.DataFrame(all_predictions)

# Aggregate predictions 
df_predictions_mean = (df_predictions_nested.groupby(["Model", "Sample_ID"]).agg({"True": "mean","Predicted": "mean"}).reset_index())

# Computes residuals (error per sample)
df_predictions_mean["Residual"] = (df_predictions_mean["True"] - df_predictions_mean["Predicted"])

# Summary from averaged predictions
summary_mean_rows = []

for model_name in df_predictions_mean["Model"].unique():
    df_m = df_predictions_mean[df_predictions_mean["Model"] == model_name]

    # Computing metrics on aggregated predictions
    summary_mean_rows.append({"Model": model_name,"r2_mean_prediction": r2_score(df_m["True"],df_m["Predicted"]),"rmse_mean_prediction": safe_rmse(df_m["True"],df_m["Predicted"]),"mae_mean_prediction": mean_absolute_error(df_m["True"],df_m["Predicted"])})

# Convert summary into a dataframe
df_summary_mean = pd.DataFrame(summary_mean_rows).sort_values(by="r2_mean_prediction",ascending=False)

# Parameter summary table
df_params = pd.DataFrame(all_params)

# Average training performance per model 
df_train_mean = (df_params.groupby("Model")[["r2_train","rmse_train","mae_train"]].mean().reset_index())

# Average test performance per model
df_test_mean = (df_params.groupby("Model")[["r2_test","rmse_test","mae_test"]].mean().reset_index())

# Feature selection tracking
df_selected_features = pd.DataFrame(all_selected_features)

# Feature frequency analysis per model
df_feature_frequency = (df_selected_features
    .groupby(["Model", "Feature"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Model", "Count"], ascending=[True, False]))
n_outer_folds = df_selected_features["Fold"].nunique()
df_feature_frequency["Selection_Frequency"] = (df_feature_frequency["Count"]/n_outer_folds)

# Feature frequency analysis per model and feature selection method
df_feature_frequency_by_selector = (df_selected_features
    .groupby(["Model", "Feature_Selection_Method", "Feature"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Model", "Feature_Selection_Method", "Count"],ascending=[True, True, False]))
df_feature_frequency_by_selector["Selection_Frequency"] = (df_feature_frequency_by_selector["Count"]/n_outer_folds)

# Frequency of feature selection methods chosen by Optuna
df_selector_frequency = (
    df_params
    .groupby(["Model", "Feature_Selection_Method"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Model", "Count"], ascending=[True, False]))
df_selector_frequency["Selection_Method_Frequency"] = (df_selector_frequency["Count"]/n_outer_folds)

## 8. Export Data to Excel
This cell should not be modified.


In [ ]:
# Saves all RAW data to different sheets in one excel document
with pd.ExcelWriter(results_dir / f"Modeling_results_{run_date}_{output_prefix}.xlsx") as writer:
    df_summary_nested.to_excel(writer,sheet_name="Summary_all_predictions",index=False)
    df_summary_mean.to_excel(writer,sheet_name="Summary_mean_predictions",index=False)
    df_predictions_nested.to_excel(writer,sheet_name="Predictions_all",index=False)
    df_predictions_mean.to_excel(writer,sheet_name="Predictions_mean",index=False)
    df_params.to_excel(writer,sheet_name="Params",index=False)
    df_train_mean.to_excel(writer,sheet_name="Train_mean",index=False)
    df_feature_frequency.to_excel(writer,sheet_name="Feature_frequency",index=False)
    df_feature_frequency_by_selector.to_excel(writer,sheet_name="Feature_freq_by_selector",index=False)
    df_selector_frequency.to_excel(writer,sheet_name="Selector_frequency",index=False)

## 9. Out of Fold Models as Plots
Ploting cell, modify only if needed


In [ ]:
# Plots the ensemble modeling (averages of all folds) of each model type
for model_name in df_predictions_mean["Model"].unique():

    # Creates dataframe
    df_m = df_predictions_mean[df_predictions_mean["Model"] == model_name]

    # Summary metrics for each model
    summary_m = df_summary_mean[df_summary_mean["Model"] == model_name].iloc[0]
    train_m = df_train_mean[df_train_mean["Model"] == model_name].iloc[0]

    # Plotting
    plt.figure(figsize=(3, 3))
    plt.scatter(df_m["True"], df_m["Predicted"], color="#98d6f4", edgecolor="#6498cc", s=20, alpha=0.7, label = "Out of Fold")
    min_val = min(df_m["True"].min(),df_m["Predicted"].min())
    max_val = max(df_m["True"].max(),df_m["Predicted"].max())
    plt.plot([min_val, max_val],[min_val, max_val],color="black",linewidth=0.5)
    plt.text(0.05,0.95,(
             f"Out of Fold R²={summary_m['r2_mean_prediction']:.3f}\n"
             f"Out of Fold MAE={summary_m['mae_mean_prediction']:.3f}"),
             transform=plt.gca().transAxes,verticalalignment="top",fontsize=6)
    plt.xlabel("Experimental ln(r)")
    plt.ylabel("Predicted ln(r)")
    plt.title(f"Ensemble {model_name} Train and Val")
    plt.legend(fontsize=6,loc="lower left",bbox_to_anchor=(0.6, 0.03))
    plt.tight_layout(rect=[0, 0, 1.5, 1.0])
    plt.xlim(-5,4)
    plt.ylim(-5,4)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xticks(np.arange(-4, 5, 1))

    # Saves figure 
    plt.savefig(results_dir / f"OutofFold_{model_name}_{output_prefix}_{run_date}.svg",dpi=300)
    plt.show()
    plt.close()

## 10. Comparison of All Models
This bar plot compares models using the validation R² (can be modified) obtained from averaged predictions
This cell should not be modified unless different metric desired to sort by




In [ ]:
# Plotting a bar chart that compares a certain metric across all model types
plt.figure(figsize=(4, 3))
ax = sns.barplot(data=df_summary_mean, x="Model", y="r2_mean_prediction", color="lightblue")
for container in ax.containers:
    ax.bar_label(container,fmt="%.2f", padding=2, fontsize=6, label_type="center")
plt.ylabel("Out of Fold R²")
plt.xticks(rotation=45)
plt.tight_layout()

# Saves figure
plt.savefig(results_dir / f"comparison_mean_{output_prefix}_{run_date}.svg",dpi=300)
plt.show()
plt.close()

# Plotting a bar chart that compares a certain metric across all model types
plt.figure(figsize=(4, 3))
ax = sns.barplot(data=df_summary_mean,x="Model",y="mae_mean_prediction",color="lightblue")
for container in ax.containers:
    ax.bar_label(container,fmt="%.2f",padding=2,fontsize=6,label_type="center")
plt.ylabel("Out of Fold MAE")
plt.xticks(rotation=45)
plt.tight_layout()

# Saves figure
plt.savefig(results_dir / f"comparison_mae_{output_prefix}_{run_date}.svg",dpi=300)
plt.show()
plt.close()

# Plot R² overfitting gap
# Positive gap = train performance is better than CV performance
# Larger positive gap = stronger possible overfitting
df_overfit = df_train_mean.merge(df_summary_mean,on="Model",how="inner")
df_overfit["Overfit_Gap_R2"] = (df_overfit["r2_train"] - df_overfit["r2_mean_prediction"])
df_overfit = df_overfit.sort_values(by="Overfit_Gap_R2", ascending=False)

# Plot overfitting gap for each model type
plt.figure(figsize=(4, 3))
ax = sns.barplot(data=df_overfit, x="Model", y="Overfit_Gap_R2", color="lightblue")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=6, label_type="center")
plt.axhline(0, color="black", linewidth=0.7)
plt.ylabel("In Fold R² - Out of Fold R²")
plt.xticks(rotation=45)
plt.tight_layout()

# Saves figure
plt.savefig(results_dir / f"overfitting_gap_{output_prefix}_{run_date}.svg",dpi=300)
plt.show()
plt.close()

## 11. Feature Selection Frequency and Identification
This cell should not be modified. It shows a bar plot of the top 20 features selected and their frequency for each model, as well as the feature selection method.




In [ ]:
# Plots the feature frequency of each model type
for model_name in df_feature_frequency["Model"].unique():

    # Creates dataframe
    df_f = df_feature_frequency[df_feature_frequency["Model"] == model_name].head(10)

    # Display dataframe
    display(df_f)

    # Plotting
    plt.figure(figsize=(12, 3))
    ax = sns.barplot(data=df_f,x="Selection_Frequency",y="Feature",color="lightblue")
    for container in ax.containers:
        ax.bar_label(container,fmt="%.2f",padding=2,fontsize=6,label_type="center")
    plt.title(f"{model_name} - Top 10 Features by Selection Frequency",fontsize=10,fontweight="bold")
    plt.tight_layout()

    # Save plot
    plt.savefig(results_dir / f"feature_frequency_{model_name}_{output_prefix}.svg",dpi=300)
    plt.show()
    plt.close()

# Plots the frequency of selected feature selection methods for each model type
for model_name in df_selector_frequency["Model"].unique():
    # Creates dataframe
    df_s = (df_selector_frequency[df_selector_frequency["Model"] == model_name].head(4))

    # Display dataframe
    display(df_s)

    # Plotting
    plt.figure(figsize=(5,4))
    ax = sns.barplot(data=df_s,x="Selection_Method_Frequency",y="Feature_Selection_Method",color="lightblue")
    for container in ax.containers:
        ax.bar_label(container,fmt="%.2f",padding=2,fontsize=6,label_type="center")
    plt.title(f"{model_name} - Feature Selection Method Frequency",fontsize=10,fontweight="bold")
    plt.tight_layout()

    # Save plot
    plt.savefig(results_dir / f"selector_frequency_{model_name}_{output_prefix}_{run_date}.svg",dpi=300)
    plt.show()
    plt.close()

## 12. List of Features per Model
This cell should not be modified. It reports the features in each model and selector type in an alphabetical list.




In [ ]:
# Reports all features used in each model type
model_features = (
    df_selected_features
    .groupby("Model")["Feature"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="All_Selected_Features")
)
pd.set_option("display.max_colwidth", None)
display(model_features)

# # Reports all features used by model and feature selection method
model_selector_features = (
    df_selected_features
    .groupby(["Model", "Feature_Selection_Method"])["Feature"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="All_Selected_Features")
)
pd.set_option("display.max_colwidth", None)
display(model_selector_features)

# # Reports the features used in each fold of each model
features_by_model_fold = (
    df_selected_features
    .groupby(["Model", "Fold", "Feature_Selection_Method"])["Feature"]
    .apply(lambda x: tuple(sorted(x)))
    .reset_index(name="Selected_Features")
)
pd.set_option("display.max_colwidth", None)
display(features_by_model_fold)

## 13. "In Sample" Evaluation of Held-Out Data (Data Leakage prevention) Plotted On Top Of Final Models



In [ ]:
# "In Sample" external dataset needs to be imported
df_is = pd.read_excel("./InputData/Data.xlsx",sheet_name="Sheet2")
# Target variable for external dataset (use same variable name as training)
y_is = df_is[target_column]
# Build X_is (in sample) by dropping the target, id columns, and any columns we dropped during training
X_is = df_is.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data; fill missing features with 0
X_is = X_is.apply(pd.to_numeric, errors="coerce")
X_is = X_is.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

# Plot each model's CV predictions with external predictions overlaid, and include metrics in the plot text
for model_name in df_predictions_mean["Model"].unique():
    df_cv = df_predictions_mean[df_predictions_mean["Model"] == model_name]
    summary_m = df_summary_mean[df_summary_mean["Model"] == model_name].iloc[0]
    train_m = df_train_mean[df_train_mean["Model"] == model_name].iloc[0]
    test_m = df_test_mean[df_test_mean["Model"] == model_name].iloc[0]

    # External predictions
    model = models_dict[model_name]
    y_pred_is = model.predict(X_is)
    r2_is = r2_score(y_is, y_pred_is)
    rmse_is = np.sqrt(np.mean((y_is - y_pred_is)**2))
    mae_is = mean_absolute_error(y_is, y_pred_is)

    # Plot
    plt.figure(figsize=(3, 3))

    # CV predictions
    plt.scatter(df_cv["True"],df_cv["Predicted"],color="#98d6f4",edgecolor="#6498cc",s=20,alpha=0.7,label="Out of Fold")

    # External predictions
    plt.scatter(y_is,y_pred_is,color="#cfb5af",edgecolor="#a76d60",s=25,alpha=0.9,label="In Sample")

    # x=y line
    min_val = min(df_cv["True"].min(),df_cv["Predicted"].min(),y_is.min(),y_pred_is.min())
    max_val = max(df_cv["True"].max(),df_cv["Predicted"].max(),y_is.max(),y_pred_is.max())
    plt.plot([min_val, max_val],[min_val, max_val],color="black",linewidth=0.5)

    # Metrics text
    plt.text(0.05, 0.95, (
             f"Out of Fold R²={summary_m['r2_mean_prediction']:.3f}\n"
             f"Out of Fold MAE={summary_m['mae_mean_prediction']:.3f}\n"
             f"In Sample R²={r2_is:.3f}\n"
             f"In Sample MAE={mae_is:.3f}\n"),transform=plt.gca().transAxes,verticalalignment="top",fontsize=6)
    plt.xlabel("Experimental ln(r)")
    plt.ylabel("Predicted ln(r)")
    plt.title(f"Ensemble {model_name} Train, Val, and In Sample Test")
    plt.legend(fontsize=6,loc="lower left",bbox_to_anchor=(0.6, 0.03))
    plt.tight_layout(rect=[0, 0, 1.5, 1.0])
    plt.xlim(-5,4)
    plt.ylim(-5,4)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xticks(np.arange(-4, 5, 1))

    # Save plots
    plt.savefig(results_dir / f"OutofFoldandInSampleModel_{model_name}_{output_prefix}_{run_date}.svg",dpi=300,bbox_inches="tight")
    plt.show()
    plt.close()

## 14. View all 178 data together on optimized models (for virtual screening) with retraining on new "ALL" spreadsheet



In [ ]:
# Whole dataset used for final model predictions
df_all = pd.read_excel("./InputData/Data.xlsx",sheet_name="Sheet3")
# Target variable (use same variable name as training)
y_all = df_all[target_column]
# Build X_all by dropping the target, id columns, and any columns we dropped during training
X_all = df_all.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data; fill missing features with 0
X_all = X_all.apply(pd.to_numeric, errors="coerce")
X_all = X_all.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

# Plot each model's CV predictions with ALL predictions overlaid, and include metrics in the plot text
for model_name in df_predictions_mean["Model"].unique():
    # ALL predictions
    model = models_dict[model_name]
    y_pred_all = model.predict(X_all)
    r2_all = r2_score(y_all, y_pred_all)
    rmse_all = np.sqrt(np.mean((y_all - y_pred_all)**2))
    mae_all = mean_absolute_error(y_all, y_pred_all)

    # Plot
    plt.figure(figsize=(3, 3))

    # ALL predictions
    plt.scatter(y_all, y_pred_all, color="#839073", edgecolor="black", s=25, alpha=0.9, label="All")

    # x=y line
    min_val = min(df_cv["True"].min(), df_cv["Predicted"].min(), y_all.min(), y_pred_all.min())
    max_val = max(df_cv["True"].max(), df_cv["Predicted"].max(), y_all.max(), y_pred_all.max())
    plt.plot([min_val, max_val],[min_val, max_val], color="black", linewidth=0.5)

    # Metrics text
    plt.text(0.05, 0.95, (
            f"R² all={r2_all:.3f}\n"
            f"MAE all={mae_all:.3f}\n"), transform=plt.gca().transAxes, verticalalignment="top", fontsize=6)
    plt.xlabel("Experimental ln(r)")
    plt.ylabel("Predicted ln(r)")
    plt.title(f"Ensemble {model_name} All Data")
    plt.legend(fontsize=6,loc="lower left",bbox_to_anchor=(0.6, 0.03))
    plt.tight_layout(rect=[0, 0, 1.5, 1.0])
    plt.xlim(-5,4)
    plt.ylim(-5,4)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xticks(np.arange(-4, 5, 1)) 

    # Save plots
    plt.savefig(results_dir / f"AllData_{model_name}_{output_prefix}_{run_date}.svg",dpi=300,bbox_inches="tight")
    plt.show()
    plt.close()
    df_all.to_excel(results_dir / f"df_AllData_{run_date}.xlsx", index=False) 

## 15. Univariate Parity Plots


In [ ]:
# Can plot univariates on the full data set or alternate data set to highlight specific experiments in the univariate plots
df_all = pd.read_excel("./InputData/Data.xlsx", sheet_name="Sheet3")
# Target variable
y_all = pd.to_numeric(df_all[target_column], errors="coerce").reset_index(drop=True)
# Build X_all by dropping the target, ID columns, and dropped training columns
X_all = df_all.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data
X_all = (X_all.apply(pd.to_numeric, errors="coerce").reindex(columns=X.columns).reset_index(drop=True))

experiment_column = "Experiment"
highlight_experiments = []
exp_ids_all = df_all[experiment_column].reset_index(drop=True)

# Features selected by each model
model_features = (df_selected_features.groupby("Model")["Feature"].apply(lambda x: sorted(x.dropna().unique())))

pd.set_option("display.max_colwidth", None)
display(model_features.reset_index(name="All_Selected_Features"))

# Output directory for saving data
univariate_dir = results_dir / f"univariate_all_data_{output_prefix}_{run_date}"
univariate_dir.mkdir(parents=True, exist_ok=True)

# In case features have weird terms that are not adaptable 
def clean_filename(name):
    name = str(name)
    name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    return name[:180]

highlight_set = {str(x) for x in highlight_experiments}
stats_rows = []

for model_name, selected_features in model_features.items():
    safe_model_name = clean_filename(model_name)
    safe_output_prefix = clean_filename(output_prefix)
    model_dir = univariate_dir / safe_model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    for feature in selected_features:
        if feature not in X_all.columns:
            print(f"Skipping {model_name} | {feature}: feature not found in X_all")
            continue

        df_uni = (pd.DataFrame({experiment_column: exp_ids_all,"Feature": X_all[feature],"lnr": y_all}).replace([np.inf, -np.inf], np.nan).dropna(subset=["Feature", "lnr"]))
        
        if df_uni.empty:
            print(f"Skipping {model_name} | {feature}: no valid values")
            continue
        safe_feature = clean_filename(feature)

        # Highlight experiments in the scatter plot
        df_uni["_exp_match"] = (df_uni[experiment_column].astype(str).str.replace(r"\.0$", "", regex=True))
        highlight_mask = df_uni["_exp_match"].isin(highlight_set)

        # Univariate fit statistics
        if len(df_uni) >= 3 and df_uni["Feature"].nunique() > 1:
            fit = linregress(df_uni["Feature"], df_uni["lnr"])
            r2, p_value = fit.rvalue ** 2, fit.pvalue
        else:
            fit, r2, p_value = None, np.nan, np.nan

        stats_rows.append({"Model": model_name, "Feature": feature,
                           "n": len(df_uni), "R2": r2, "p_value": p_value})

        stats_text = "R² = NA\np = NA" if np.isnan(r2) else f"R² = {r2:.3f}\np = {p_value:.3g}"

        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

        # Scatter plot of all non-highlighted points
        axes[0].scatter(df_uni.loc[~highlight_mask, "Feature"], df_uni.loc[~highlight_mask, "lnr"], color="#98d6f4", edgecolor="#6498cc",s=25,alpha=0.9,label="Other experiments")

        if highlight_mask.any():
            axes[0].scatter(df_uni.loc[highlight_mask, "Feature"], df_uni.loc[highlight_mask, "lnr"],
                            color="#839073", edgecolor="#0d6b30", s=80, marker="*", alpha=1.0,
                            zorder=5, label="Highlighted experiments")
            axes[0].legend(fontsize=7, loc="lower right")

        # Fit line + stats text
        if fit is not None:
            xs = np.array([df_uni["Feature"].min(), df_uni["Feature"].max()])
            axes[0].plot(xs, fit.intercept + fit.slope * xs, color="k", lw=1)
        axes[0].text(0.03, 0.97, stats_text, transform=axes[0].transAxes, va="top", ha="left", fontsize=8)

        # Scatter plot specifications
        axes[0].set_xlabel(textwrap.fill(feature, width=35), fontsize=7)
        axes[0].set_ylabel("Experimental ln(r)")
        axes[0].set_title("Univariate Fit", fontsize=8)
        axes[0].set_ylim(-6, 5)
        axes[0].set_yticks(np.arange(-4, 5, 1))

        # Histogram plot specifications
        axes[1].hist(df_uni["Feature"], bins=15, color="#98d6f4", edgecolor="#6498cc", alpha=0.9)
        for x_val in df_uni.loc[highlight_mask, "Feature"]:
            axes[1].axvline(x_val, color="#0d6b30", linewidth=1.2, alpha=0.9)
        axes[1].set_xlabel(textwrap.fill(feature, width=35), fontsize=7)
        axes[1].set_ylabel("Frequency")
        axes[1].set_title("Univariate Distribution", fontsize=8)

        # Overall figure specifications
        fig.suptitle(f"{model_name}: {feature}", fontsize=9)
        fig.tight_layout()
        fig.savefig(model_dir / f"univariate_{safe_model_name}_{safe_feature}_{safe_output_prefix}_{run_date}.svg", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

df_stats = pd.DataFrame(stats_rows)
df_stats.to_excel(results_dir / f"Univariates_{run_date}.xlsx", index=False)

## 16. xyplots


In [ ]:
# Need to run univariate cell first to generate df_selected_features before running this cell
# Choose "selected_by_model" to make XY plots for selected features within each model, choose "manual_lists" to use features_1 and features_2 directly, independent of model
xy_mode = "selected_by_model"

# For xy_mode = "selected_by_model": None = use all selected features for each model
# For xy_mode = "manual_lists": None or "all_features" = use all X_all columns
# Otherwise, use a list of feature names or X_all column positions
features_1 = None
features_2 = None

# Optional: restrict which features can appear in selected_by_model plots (None = no restriction)
xy_features_to_plot = None

# R^2 cutoff below which the feature pair will be recorded but not plotted
r2_cutoff = 0.0

# Avoid making both A vs B and B vs A
skip_duplicate_reverse_pairs = True

# Limit number of plots per model/manual group; use None for no limit
max_pairs_per_group = None

# Plot appearance
color_by_response = True
response_label = "Experimental ln(r)"
response_vmin, response_vmax = -4, 4
response_cmap = LinearSegmentedColormap.from_list("custom", ["#f5f5f5", "#98d6f4", "#6498cc", "#003554", "#6498cc", "#98d6f4", "#f5f5f5"])

# Highlight specific experiments 
experiment_column = "Experiment"
highlight_experiments = []
show_plots = True
save_plots = True

# Rebuild model_features in case this cell is run independently after df_selected_features exists
model_features = (df_selected_features.groupby("Model")["Feature"].apply(lambda x: sorted(x.dropna().unique())))

# Output directory for saving XY plots
safe_output_prefix = clean_filename(output_prefix)
xy_dir = results_dir / f"XYplots_{safe_output_prefix}_{run_date}"
xy_dir.mkdir(parents=True, exist_ok=True)

# Experiment IDs for highlighting
if experiment_column in df_all.columns:
    exp_ids_all = df_all[experiment_column].reset_index(drop=True)
else:
    print(f"Warning: {experiment_column} not found in df_all. Using row numbers instead.")
    exp_ids_all = pd.Series(np.arange(len(df_all)), name=experiment_column)

highlight_set = {str(x) for x in highlight_experiments}

def resolve_xy_features(feature_spec, X_reference):
    if feature_spec is None:
        return list(X_reference.columns)
    if isinstance(feature_spec, str):
        if feature_spec.lower() in {"all_features", "all", "x_all"}:
            return list(X_reference.columns)
        feature_spec = [feature_spec]
    resolved_features = []
    for feature in feature_spec:
        if isinstance(feature, (int, np.integer)):
            feature_index = int(feature)
            if -len(X_reference.columns) <= feature_index < len(X_reference.columns):
                resolved_features.append(X_reference.columns[feature_index])
            else:
                print(f"Skipping column index {feature}: out of range for X_all")
        elif feature in X_reference.columns:
            resolved_features.append(feature)
        else:
            print(f"Skipping {feature}: feature not found in X_all")

    # Preserve order while removing duplicates
    return list(dict.fromkeys(resolved_features))

def get_xy_pairs(features_a, features_b, skip_reverse=True):
    seen_pairs = set()
    for feature_a, feature_b in itertools.product(features_a, features_b):
        if feature_a == feature_b:
            continue
        if skip_reverse:
            pair_key = tuple(sorted([feature_a, feature_b]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
        yield feature_a, feature_b

def plot_xy_pair(model_name, feature_1, feature_2, model_dir):
    for feat in (feature_1, feature_2):
        if feat not in X_all.columns:
            print(f"Skipping {model_name} | {feat}: feature not found in X_all")
            return None

    df_xy = (pd.DataFrame({experiment_column: exp_ids_all,
                           "Feature_1": X_all[feature_1],
                           "Feature_2": X_all[feature_2],
                           "lnr": y_all})
             .replace([np.inf, -np.inf], np.nan))
    required_cols = ["Feature_1", "Feature_2"] + (["lnr"] if color_by_response else [])
    df_xy = df_xy.dropna(subset=required_cols)

    if df_xy.empty:
        print(f"Skipping {model_name} | {feature_1} vs {feature_2}: no valid values")
        return None

    # Feature 1 vs Feature 2 fit statistics
    if len(df_xy) >= 3 and df_xy["Feature_1"].nunique() > 1 and df_xy["Feature_2"].nunique() > 1:
        fit = linregress(df_xy["Feature_1"], df_xy["Feature_2"])
        r2, p_value = fit.rvalue ** 2, fit.pvalue
        slope, intercept = fit.slope, fit.intercept
    else:
        fit, r2, p_value, slope, intercept = None, np.nan, np.nan, np.nan, np.nan

    record = {"Model": model_name, "Feature_1": feature_1, "Feature_2": feature_2,
              "n": len(df_xy), "R2": r2, "p_value": p_value,
              "slope": slope, "intercept": intercept, "Plotted": False}

    if np.isnan(r2):
        print(f"Skipping plot for {model_name} | {feature_1} vs {feature_2}: not enough variance/points")
        return record
    if r2 < r2_cutoff:
        print(f"R² below cutoff. Skipping plot for {model_name} | {feature_1} vs {feature_2}: "
              f"R² = {r2:.3f}, p = {p_value:.3g}")
        return record

    # Highlight experiments
    df_xy["_exp_match"] = (df_xy[experiment_column].astype(str).str.replace(r"\.0$", "", regex=True))
    highlight_mask = df_xy["_exp_match"].isin(highlight_set)

    stats_text = f"R² = {r2:.3f}\np = {p_value:.3g}"

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

    # Panel 0: Feature 1 vs Feature 2 scatter + fit line
    if color_by_response:
        sc = axes[0].scatter(df_xy["Feature_1"], df_xy["Feature_2"], c=df_xy["lnr"],
                             cmap=response_cmap, vmin=response_vmin, vmax=response_vmax,
                             edgecolor="#cfcfcf", s=25, alpha=0.9)
        cbar = fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04)
        cbar.set_label(response_label, fontsize=7)
        cbar.set_ticks(np.arange(response_vmin, response_vmax + 1, 1))
        cbar.ax.tick_params(labelsize=7)
    else:
        axes[0].scatter(df_xy.loc[~highlight_mask, "Feature_1"], df_xy.loc[~highlight_mask, "Feature_2"],
                        color="#98d6f4", edgecolor="#6498cc", s=25, alpha=0.9, label="Other experiments")

    if highlight_mask.any():
        axes[0].scatter(df_xy.loc[highlight_mask, "Feature_1"], df_xy.loc[highlight_mask, "Feature_2"],
                        color="#839073", edgecolor="#0d6b30", s=80, marker="*", alpha=1.0,
                        zorder=5, label="Highlighted experiments")
        axes[0].legend(fontsize=7, loc="lower right")

    # Fit line + stats text
    xs = np.array([df_xy["Feature_1"].min(), df_xy["Feature_1"].max()])
    axes[0].plot(xs, intercept + slope * xs, color="k", lw=1)
    axes[0].text(0.03, 0.97, stats_text, transform=axes[0].transAxes, va="top", ha="left",
                 fontsize=8, bbox=dict(facecolor="white", edgecolor="none", alpha=0.75))

    # Scatter plot specifications
    axes[0].set_xlabel(textwrap.fill(feature_1, width=35), fontsize=7)
    axes[0].set_ylabel(textwrap.fill(feature_2, width=35), fontsize=7)
    axes[0].set_title("Feature-Feature Fit", fontsize=8)

    # Panels 1 and 2: histograms of each feature
    for ax, col, label, title in [(axes[1], "Feature_1", feature_1, "Feature 1 Distribution"),
                                  (axes[2], "Feature_2", feature_2, "Feature 2 Distribution")]:
        ax.hist(df_xy[col], bins=15, color="#98d6f4", edgecolor="#6498cc", alpha=0.9)
        for val in df_xy.loc[highlight_mask, col]:
            ax.axvline(val, color="#0d6b30", linewidth=1.2, alpha=0.9)
        ax.set_xlabel(textwrap.fill(label, width=35), fontsize=7)
        ax.set_ylabel("Frequency")
        ax.set_title(title, fontsize=8)

    # Overall figure specifications
    fig.suptitle(f"{model_name}: {feature_1} vs {feature_2}", fontsize=9)
    fig.tight_layout()

    if save_plots:
        safe_model_name = clean_filename(model_name)
        safe_feature_1 = clean_filename(feature_1)[:80]
        safe_feature_2 = clean_filename(feature_2)[:80]
        fig.savefig(model_dir / f"xy_{safe_model_name}_{safe_feature_1}_vs_{safe_feature_2}_{safe_output_prefix}_{run_date}.svg",
                    dpi=300, bbox_inches="tight")
    if show_plots:
        plt.show()
    plt.close(fig)

    record["Plotted"] = True
    return record

# Build plotting jobs
xy_jobs = []
if xy_mode == "selected_by_model":
    xy_features_to_plot_resolved = None
    if xy_features_to_plot is not None:
        xy_features_to_plot_resolved = set(resolve_xy_features(xy_features_to_plot, X_all))

    for model_name, selected_features in model_features.items():
        selected_features = [f for f in selected_features if f in X_all.columns]
        if xy_features_to_plot_resolved is not None:
            selected_features = [f for f in selected_features if f in xy_features_to_plot_resolved]
        if len(selected_features) < 2:
            print(f"Skipping {model_name}: fewer than 2 selected features available for XY plots")
            continue

        resolved_features_1 = (selected_features if features_1 is None
                               else [f for f in resolve_xy_features(features_1, X_all) if f in selected_features])
        resolved_features_2 = (selected_features if features_2 is None
                               else [f for f in resolve_xy_features(features_2, X_all) if f in selected_features])
        if len(resolved_features_1) == 0 or len(resolved_features_2) == 0:
            print(f"Skipping {model_name}: no matching requested features found among selected features")
            continue
        xy_jobs.append((model_name, resolved_features_1, resolved_features_2))

elif xy_mode == "manual_lists":
    manual_features_1 = resolve_xy_features(features_1, X_all)
    manual_features_2 = resolve_xy_features(features_2, X_all)
    xy_jobs.append(("manual_feature_lists", manual_features_1, manual_features_2))
else:
    raise ValueError("xy_mode must be either 'selected_by_model' or 'manual_lists'")

# Run plots and collect stats
xy_stats_records = []
for model_name, resolved_features_1, resolved_features_2 in xy_jobs:
    safe_model_name = clean_filename(model_name)
    model_dir = xy_dir / safe_model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    pair_counter = 0
    for feature_1, feature_2 in get_xy_pairs(resolved_features_1, resolved_features_2,
                                             skip_reverse=skip_duplicate_reverse_pairs):
        if max_pairs_per_group is not None and pair_counter >= max_pairs_per_group:
            print(f"Reached max_pairs_per_group = {max_pairs_per_group} for {model_name}")
            break
        pair_counter += 1
        record = plot_xy_pair(model_name, feature_1, feature_2, model_dir)
        if record is not None:
            xy_stats_records.append(record)

# Save stats table
if len(xy_stats_records) > 0:
    df_xy_stats = (pd.DataFrame(xy_stats_records)
                   .sort_values(by=["Model", "R2"], ascending=[True, False], na_position="last")
                   .reset_index(drop=True))
    display(df_xy_stats)
    stats_path = results_dir / f"XY_correlations_{run_date}.xlsx"
    df_xy_stats.to_excel(stats_path, index=False)
    print(f"Saved XY correlation stats to: {stats_path}")
else:
    print("No XY feature pairs were plotted or recorded.")

## 17. Leave One Class Out


In [ ]:
# Leave one class out analysis for each monomer class (in either position M1 or M2) using the final model trained on all data
FINAL_MODEL = "KRR"
# Name of columns that contain the class labels for each experiment; used for leave-one-class-out (LOCO) analysis
CLASS_COLUMNS = ["M1", "M2"]   
MIN_CLASS_SIZE = 1  
AXIS_LIMITS = (-5, 4)
SHOW_PLOTS = True
SAVE_PLOTS = True

# Features/target aligned to the training columns (errors out instead of silently filling 0)
missing = [c for c in list(X.columns) + CLASS_COLUMNS if c not in df_all.columns]
if missing:
    raise ValueError(f"df_all is missing {len(missing)} required column(s): {missing[:5]}")
X_data = df_all[X.columns].apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
y_data = pd.to_numeric(df_all[target_column], errors="coerce").reset_index(drop=True)

# Build the classes automatically: an experiment belongs to a class if the class appears in M1 or M2
labels = (df_all[CLASS_COLUMNS]
          .apply(lambda s: s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True))
          .where(df_all[CLASS_COLUMNS].notna())
          .reset_index(drop=True))
class_names = sorted(pd.unique(labels.stack()))
loco_masks = {cls: labels.eq(cls).any(axis=1).to_numpy() for cls in class_names}
print(f"{len(class_names)} classes found: {class_names}")

model = models_dict[FINAL_MODEL]
loco_rows = []

for cls, mask in loco_masks.items():
    if mask.sum() < MIN_CLASS_SIZE or mask.all():
        print(f"Skipping class {cls}: {mask.sum()} held-out experiments")
        continue

    X_loco, y_loco = X_data[mask], y_data[mask]
    X_rest, y_rest = X_data[~mask], y_data[~mask]
    pred_loco, pred_rest = model.predict(X_loco), model.predict(X_rest)

    r2_loco, mae_loco = r2_score(y_loco, pred_loco), mean_absolute_error(y_loco, pred_loco)
    r2_rest, mae_rest = r2_score(y_rest, pred_rest), mean_absolute_error(y_rest, pred_rest)
    loco_rows.append({"Class": cls, "n_heldout": int(mask.sum()), "R2_LOCO": r2_loco, "MAE_LOCO": mae_loco,
                      "R2_rest": r2_rest, "MAE_rest": mae_rest})

    fig, ax = plt.subplots(figsize=(3, 3))
    ax.scatter(y_rest, pred_rest, color="#98d6f4", edgecolor="#6498cc", s=20, alpha=0.7,
               label="All data except LOMO")
    ax.scatter(y_loco, pred_loco, color="#cfb5af", edgecolor="#a76d60", s=25, alpha=0.9, label="LOMO")
    ax.plot(AXIS_LIMITS, AXIS_LIMITS, color="black", linewidth=0.5)
    ax.text(0.05, 0.95, (f"R² non-LOMO = {r2_rest:.3f}\nMAE non-LOMO = {mae_rest:.3f}\n"
                         f"R² LOMO = {r2_loco:.3f}\nMAE LOMO = {mae_loco:.3f}"),
            transform=ax.transAxes, va="top", fontsize=6)
    ax.set_xlabel("Experimental ln(r)")
    ax.set_ylabel("Predicted ln(r)")
    ax.set_title(f"Ensemble {FINAL_MODEL} | held out {cls}")
    ax.set_xlim(AXIS_LIMITS)
    ax.set_ylim(AXIS_LIMITS)
    ax.set_xticks(np.arange(-4, 5, 1))
    ax.set_aspect("equal", adjustable="box")
    ax.legend(fontsize=6, loc="lower right")

    if SAVE_PLOTS:
        fig.savefig(results_dir / f"LOCO_{clean_filename(cls)}_{FINAL_MODEL}_{output_prefix}_{run_date}.svg",
                    bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

df_loco = pd.DataFrame(loco_rows)
display(df_loco)
df_loco.to_excel(results_dir / f"LOCO_stats_{FINAL_MODEL}_{output_prefix}_{run_date}.xlsx", index=False)

## 18. Prediction parity plot between top 2 model types


In [ ]:
model1 = models_dict["KRR"]
model2 = models_dict["XGB"]
y_pred_all1 = model1.predict(X_all)
y_pred_all2 = model2.predict(X_all)

plt.figure(figsize=(3, 3))

plt.scatter(y_pred_all1, y_pred_all2, color="purple",edgecolor="black",s=25,alpha=0.9,label="ALL")
plt.plot([min_val, max_val],[min_val, max_val],color="black",linewidth=0.5)
plt.xlabel("KRR predictions")
plt.ylabel("XGB predictions")
plt.title(f"KRR vs XGB - ALL predictions")
plt.tight_layout(rect=[0, 0, 1.5, 1.0])
r2_krr_xgb = r2_score(y_pred_all1, y_pred_all2)
plt.text(0.05,0.95,f"R²={r2_krr_xgb:.2f}",transform=plt.gca().transAxes,verticalalignment="top",fontsize=6)
plt.savefig(results_dir / f"KRR_vs_XGB_ALL_{output_prefix}.svg",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

## 19. Shap analysis on selected model types


In [ ]:
for model_name in df_predictions_mean["Model"].unique():

        if model_name not in ["KRR"]:
                continue

        model = models_dict[model_name]
        print(f"Running SHAP for model: {model_name}")

        # Shap
        X_train_model = X
        X_full_model = X_all

        # Use small background sample
        background = shap.sample(X_train_model, 100)

        # Create explainer
        explainer = shap.Explainer(model.predict, background)

        # Compute SHAP values
        shap_values = explainer.shap_values(X_full_model)

        # SHAP summary plot
        SHAP_MAX_DISPLAY = 3
        SHAP_PLOT_SIZE = (6, 3)

        shap.summary_plot(shap_values, X_full_model, feature_names=X_full_model.columns, max_display=SHAP_MAX_DISPLAY, plot_size=SHAP_PLOT_SIZE, show=False)
        plt.title(f"SHAP Summary - {model_name}")
        # Save SHAP plot
        plt.savefig(results_dir / f"shap_summary_{model_name}_{output_prefix}_{run_date}.png", dpi=300, bbox_inches="tight", transparent=True)

        plt.show()
        plt.close()

In [ ]:
# Used to rename features in SHAP summary plot for publication
rename_dict = {'M1_distance_CC2_CC2prime(Å)_low_E-M2_distance_CC2_CC2prime(Å)_low_E': 'Δ Distance olefin LEC',
               'M1_Hirsh_charge_C1prime_low_E-M2_Hirsh_charge_C1prime_low_E': 'Δ Hirsh charge C1prime LEC',
               'M1_Buried_Sterimol_L_CC2prime_CC2_5.0(Å)_min': 'A Buried Sterimol L olefin 5.0Å min',}

pretty_feature_names = [rename_dict.get(f, f) for f in X_full_model.columns]
shap.summary_plot(shap_values,X_full_model,feature_names=pretty_feature_names,max_display=SHAP_MAX_DISPLAY,plot_size=(10, 3),cmap=plt.get_cmap("coolwarm"),show=False)
plt.savefig(results_dir / f"manuscript_shap_summary_{model_name}_{output_prefix}.svg", dpi=300, bbox_inches="tight", transparent=True)

## 20. Virtual Screening on M1=M2 data
Input new spreadsheet with desired experiments to screen


In [ ]:
# Input virtual screening prediction spreadsheet
df_vs1 = pd.read_excel("./InputData/Data.xlsx", sheet_name="Sheet4")
# Build X_vs1 by dropping the target, id columns, and any columns we dropped during training
X_vs1 = df_vs1.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data; fill missing features with 0
X_vs1 = X_vs1.apply(pd.to_numeric, errors="coerce")
X_vs1 = X_vs1.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

# Select specific model to run virtual screening on
for model_name in df_predictions_mean["Model"].unique():
    if model_name in ["KRR"]:
        model = models_dict[model_name]
        print(f"Predicting virtual screening results with model: {model_name}")
        y_vs1_pred = model.predict(X_vs1)
        df_vs1[f"Predicted_ln_r_{model_name}"] = y_vs1_pred
        df_vs1.to_excel(results_dir / f"VirtualScreening_M1eqM2_{model_name}_{run_date}.xlsx", index=False)

        # Plotting
        plt.figure(figsize=(3,3))
        plt.hist(y_all, bins=20, alpha=0.6, color="#839073", label= "Experimental")
        plt.hist(y_vs1_pred,bins=20, alpha=0.6, color="#a76d60", label=f"M1=M2 predictions")
        plt.xlabel("ln(r)")
        plt.ylabel("Frequency")
        plt.xlim(-4,4)
        plt.xticks(np.arange(-4, 5, 1))
        plt.ylim(0,25)
        plt.title(f"{model_name}")
        plt.legend(fontsize=6,loc="upper left")
        plt.axvline(x=0.5, color="black")
        plt.axvline(x=-0.5, color="black")

plt.savefig(results_dir / f"VirtualScreening_M1eqM2Histogram_{model_name}_{run_date}.svg", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

## 21. Virtual Screening on Out of Sample Data (unseen monomer pairs)
Input new spreadsheet with desired experiments to screen


In [ ]:
# Input virtual screening prediction spreadsheet
df_vs2 = pd.read_excel("./InputData/Data.xlsx", sheet_name="Sheet5")
# Build X_vs by dropping the target, id columns, and any columns we dropped during training
X_vs2 = df_vs2.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data; fill missing features with 0
X_vs2 = X_vs2.apply(pd.to_numeric, errors="coerce")
X_vs2 = X_vs2.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

# Select specific model to run virtual screening on
for model_name in df_predictions_mean["Model"].unique():
    if model_name in ["KRR"]:
        model = models_dict[model_name]
        print(f"Predicting virtual screening results with model: {model_name}")
        y_vs2_pred = model.predict(X_vs2)
        df_vs2[f"Predicted_ln_r_{model_name}"] = y_vs2_pred
        df_vs2.to_excel(results_dir / f"VirtualScreening_OutofSamplePredictions_{model_name}_{run_date}.xlsx", index=False)

        # Plotting
        plt.figure(figsize=(3,3))
        plt.hist(y_all, bins=20, alpha=0.6, color="#6498cc", label="Experimental")
        plt.hist(y_vs2_pred, bins=20, alpha=0.6, color="#839073", label=f"VS predictions OOS {model_name}")
        plt.xlabel("ln(r)")
        plt.ylabel("Frequency")
        plt.xlim(-5,5)
        plt.xticks(np.arange(-5, 6, 1))
        plt.ylim(0,25)
        plt.title(f"{model_name}")
        plt.legend(fontsize=6,loc="upper right")
plt.savefig(results_dir / f"VirtualScreening_OutofSamplePredictionsHistogram_{model_name}_{run_date}.svg", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

## 22. Out of Sample Evaluation of Completely Unseen Data Plotted On Top Of Final Models (External Test)


In [ ]:
# External dataset needs to be imported
df_oos1 = pd.read_excel("./InputData/Data.xlsx",sheet_name="Sheet5")
# Target variable for external dataset (use same variable name as training)
y_oos1 = df_oos1[target_column]
# Build X_oos1 by dropping the target, id columns, and any columns we dropped during training
X_oos1 = df_oos1.drop(columns=[target_column] + id_columns + drop_columns, errors="ignore")
# Ensure numeric and align columns/order with training data; fill missing features with 0
X_oos1 = X_oos1.apply(pd.to_numeric, errors="coerce")
X_oos1 = X_oos1.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

# Select specific model to run out of sample evaluation on
for model_name in ["KRR"]:

    model = models_dict[model_name]
    y_pred_oos1 = model.predict(X_oos1)
    r2_oos1 = r2_score(y_oos1, y_pred_oos1)
    rmse_oos1 = np.sqrt(np.mean((y_oos1 - y_pred_oos1)**2))
    mae_oos1 = mean_absolute_error(y_oos1, y_pred_oos1)

    y_pred_all = model.predict(X_all)
    r2_all = r2_score(y_all, y_pred_all)
    rmse_all = np.sqrt(np.mean((y_all - y_pred_all)**2))
    mae_all = mean_absolute_error(y_all, y_pred_all)

    # Plot
    plt.figure(figsize=(3, 3))

    # CV predictions
    plt.scatter(y_all,y_pred_all,color="#98d6f4",edgecolor="#6498cc",s=25,alpha=0.9,label="ALL")

    # External predictions
    plt.scatter(y_oos1,y_pred_oos1,color="#839073",edgecolor="#0d6b30",s=20,alpha=0.9,label="OOS")

    # x=y line
    min_val = min(y_all.min(), y_pred_all.min(), y_oos1.min(), y_pred_oos1.min())
    max_val = max(y_all.max(), y_pred_all.max(), y_oos1.max(), y_pred_oos1.max())
    plt.plot([min_val, max_val],[min_val, max_val], color="black", linewidth=0.5)

    # Metrics text
    plt.text(0.05,0.95,(
            f"R² all={r2_all:.3f}\n"
            f"MAE all={mae_all:.3f}\n"
            f"R² oos={r2_oos1:.3f}\n"
            f"MAE oos={mae_oos1:.3f}\n"), transform=plt.gca().transAxes, verticalalignment="top", fontsize=6)
    plt.xlabel("Experimental ln(r)")
    plt.ylabel("Predicted ln(r)")
    plt.title(f"{model_name} OOS External Test")
    plt.legend(fontsize=6, loc="lower left", bbox_to_anchor=(0.6, 0.03))
    plt.tight_layout(rect=[0, 0, 1.5, 1.0])
    plt.xlim(-5, 4)
    plt.ylim(-5, 4)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xticks(np.arange(-4, 5, 1))

    # Save plots
    plt.savefig(results_dir / f"ExternalTest_{model_name}_{output_prefix}_{run_date}.svg",dpi=300,bbox_inches="tight")
    plt.show()
    plt.close()